### Working with bigger data, algorithms and out-of-core learning
out-of-core-learning lets us to work with large datasets, by fitting classifier incrementally on smaller batches of a dataset
using partial fit form SGD, to stream documents directly from local drive and train a logistic regression model using a small mini-batch of documents


In [36]:
# define a tokenizer function to clean up unprosed data
import numpy as np
import re
from nltk.corpus import stopwords

stop = stopwords.words('english')

def tokenizer(text):
    # try to remove all hmtl markup
    text = re.sub('<[^>]*>', '', text)

    # find all emotiocons
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # remove all non-word char from text via regex [\W] + convert text into lowercase
    # added temporary stored emoticons to the end of the setence
    text = (re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))

    tokenized = [w for w in text.split() if w not in stop]

    return tokenized

In [37]:
# generator function to read and returns one document at a time
def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        # skip header
        next(csv)
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label


# testin stream docs on fist line in movie csv
# should return tuple consiting of review text and class label
path = "data/movie_data.csv"

next(stream_docs(path))

('"If your idea of entertainment is watching graphic footage of people being run over by cars (you get to see a woman passing under the front wheel, being twisted as the car passes over her before she goes under the back wheel -- and they show it twice in case you missed it the first time) then this is the documentary for you. Admitedly I didn\'t watch any more of this very disturbing piece of voyeurism, but that was enough for me. Maybe the rest is even better.<br /><br />I wonder how long it\'s going to take for television networks to start showing slush movies. Perhaps game shows based on self-mutilation might be nice.<br /><br />I already know that there are disturbed people in the world and that horrible things happen. I don\'t need to see the proof on the TV masquerading as entertainment."',
 0)

In [38]:
# define function to get mini-batch
# take stream for stream docs, and return a particular number of documents specified by sixe parameter

def get_minibatch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)

    except StopIteration:
        return None, None

    return docs, y

In [39]:
# can't use tf-idf or count vectorizer since they need to hold all of the vocubalry (know it)
# but can use Hashing vectorizer

# set high number of features in hashing vectrorizer, reduce chances of hashing crashes, but also increase coefficents in our loreg
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

vect = HashingVectorizer(decode_error='ignore',
                         n_features=2**21,
                         preprocessor=None,
                         tokenizer=tokenizer)

clf = SGDClassifier(loss='log_loss', random_state=1)

doc_stream = stream_docs(path)

In [40]:
# can start the out of core learning

# 45 mini batches of documents, where each mini batch consists of 1000 documents
import pyprind

pbar = pyprind.ProgBar(45)

classes = np.array([0,1])

for _ in range(45):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)
    pbar.update()

In [41]:
# use that last 5000 to test
X_test, y_test = get_minibatch(doc_stream, 5000)

X_test = vect.transform(X_test)
print('Accuracy: %.3f' % clf.score(X_test, y_test))

Accuracy: 0.873


In [42]:
# slightly lower accuracy that the grid seatch in /sentimient_analysis.ipynb, but much more memory efficent
# can use the last 5000 docuemtns to update the clasifier
clf = clf.partial_fit(X_test, y_test)